### Chatbots With LangGraph

In [ ]:
!pip install langgraph langsmith

In [ ]:
!pip install langchain langchain_groq langchain_community

In [ ]:
import os

## Get the API keys from the file
groq_api_key = os.getenv('groq_api_key')
langsmith = os.getenv('LANGSMITH_API_KEY')

print(langsmith)

## Get the langsmith tracking running
os.environ["LANGCHAIN_API_KEY"] = langsmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "LangGraphTesting"

In [ ]:
from langchain_groq import ChatGroq

## Get the LLM Groq running well
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key = groq_api_key,
    temperature = 0
)

llm

### Building Chatbots Using Langgraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
  # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
  messages:Annotated[list, add_messages]

graph_builder = StateGraph(State)
graph_builder

In [ ]:
def chatbot(state:State):
  return {"messages":llm.invoke(state['messages'])}

graph_builder.add_node("chatbot", chatbot)

## Add edges to the chatbot
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

graph_builder

In [ ]:
from IPython.display import Image, display

## Get the graph image after compiling
graph = graph_builder.compile()

try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
  pass

In [ ]:
while True:
  user_input=input("User: ")
  if user_input.lower() in ["quit","q"]:
    print("Good Bye")
    break
  for event in graph.stream({'messages':("user",user_input)}):
    print(event.values())
    for value in event.values():
      print(value['messages'])
      print("Assistant:",value["messages"].content)